In [ ]:
# @title 1. Setup & Installation
# 1. Update the package lists first to fix the 404 error
!apt-get update > /dev/null

# 2. Install poppler-utils now that the system has the right download links
!apt-get install -y poppler-utils > /dev/null

# 3. Install the Python libraries
!pip install pdf2image pymupdf easyocr opencv-python-headless > /dev/null

print("✅ Environment ready.")

In [ ]:
# @title 2. Upload PDF
from google.colab import files
import os

uploaded = files.upload()
pdf_filename = next(iter(uploaded))
OUTPUT_DIR = "processed_data"
DPI = 300 # Standard for AEC research

print(f"✅ Loaded: {pdf_filename}")

In [ ]:
# @title 3. Helper Functions (Vector + Geometry)
import fitz # PyMuPDF
import json
import shutil

def create_folders(base):
    if os.path.exists(base): shutil.rmtree(base)
    os.makedirs(f"{base}/page_images", exist_ok=True)
    os.makedirs(f"{base}/vector_data", exist_ok=True)

def extract_vector_and_geometry(page, page_num):
    # A. Precise Text Extraction
    text_items = []
    for w in page.get_text("words"):
        text_items.append({
            "type": "vector_text", "content": w[4],
            "bbox": [round(w[0], 2), round(w[1], 2), round(w[2], 2), round(w[3], 2)]
        })

    # B. Geometric Path Extraction (Walls/Doors)
    geom_items = []
    for shape in page.get_drawings():
        if shape["rect"].width > page.rect.width * 0.9: continue
        geom_items.append({
            "type": "geometry_path",
            "bbox": [round(shape["rect"].x0, 2), round(shape["rect"].y0, 2),
                     round(shape["rect"].x1, 2), round(shape["rect"].y1, 2)]
        })

    return {
        "page": page_num,
        "dims": {"w": round(page.rect.width, 2), "h": round(page.rect.height, 2)},
        "text": text_items,
        "geometry": geom_items
    }

In [ ]:
# @title 4. Computer Vision Logic (OCR & Line Detection)
import easyocr
import cv2
import numpy as np

# Initialize AI Reader
reader = easyocr.Reader(['en'])

def run_computer_vision(image_path):
    # 1. OCR for missing/sideways labels
    ocr_results = reader.readtext(image_path)
    ocr_data = []
    for (bbox, text, prob) in ocr_results:
        xs, ys = [p[0] for p in bbox], [p[1] for p in bbox]
        ocr_data.append({
            "content": text, "conf": float(prob),
            "bbox": [int(min(xs)), int(min(ys)), int(max(xs)), int(max(ys))]
        })

    # 2. Line Detection for rasterized walls
    img_cv = cv2.imread(image_path, 0)
    edges = cv2.Canny(img_cv, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, 100, minLineLength=50, maxLineGap=10)
    line_data = []
    if lines is not None:
        for l in lines:
            line_data.append({"start": [int(l[0][0]), int(l[0][1])], "end": [int(l[0][2]), int(l[0][3])]})

    return ocr_data, line_data

In [ ]:
# @title 5. Execute Task 1 Pipeline
from pdf2image import convert_from_path

create_folders(OUTPUT_DIR)
doc = fitz.open(pdf_filename)
images = convert_from_path(pdf_filename, dpi=DPI)

for i, page in enumerate(doc):
    base_name = f"Sheet_{i+1:02d}_{pdf_filename.replace('.pdf','')}"
    img_path = f"{OUTPUT_DIR}/page_images/{base_name}.png"

    # 1. Save Image & Extract Vector Data
    images[i].save(img_path, "PNG")
    v_data = extract_vector_and_geometry(page, i+1)

    # 2. Run Vision Perception on the Image
    ocr_data, line_data = run_computer_vision(img_path)

    # 3. Combine into Final JSON
    final_output = {**v_data, "raster_ocr": ocr_data, "detected_lines": line_data}

    with open(f"{OUTPUT_DIR}/vector_data/{base_name}.json", "w") as f:
        json.dump(final_output, f, indent=4)

    print(f"✅ Processed Page {i+1}")

# Zip and download
shutil.make_archive('Task1_Complete_Dataset', 'zip', OUTPUT_DIR)
files.download('Task1_Complete_Dataset.zip')